# 05 · LLM 推理工程：让模型更快更省

> 一个 70B 参数的模型在 A100 上生成一个 token 需要多久？如何把延迟从秒级压到毫秒级？
> 本章覆盖生产环境中最核心的推理优化技术。

| 你将学到 | 关键词 |
|---------|-------|
| 为什么推理比训练更难优化 | 内存带宽瓶颈、Roofline 模型 |
| KV Cache 如何消除重复计算 | KV Cache、增量解码 |
| Flash Attention 的 IO 感知设计 | Tiling、SRAM vs HBM |
| 量化如何压缩模型体积 | INT8、INT4、bitsandbytes |
| 投机采样如何并行验证 | Speculative Decoding、Draft Model |
| 连续批处理如何提升吞吐量 | Continuous Batching、vLLM |


## 第一章：为什么推理优化至关重要？

训练是一次性成本，推理是持续运营成本。ChatGPT 每天处理数千万次请求，推理成本是训练成本的数十倍。

**核心瓶颈：内存带宽，而非算力**

LLM 推理是「内存带宽受限」（Memory-Bound）问题：每生成一个 token，需要把所有模型权重从显存（HBM）搬运到计算单元（SRAM）。

```mermaid
flowchart LR
    subgraph 训练["训练（算力受限）"]
        T1["大 batch\n高 GPU 利用率\n算术密度高"]
    end
    subgraph 推理["推理（带宽受限）"]
        I1["batch=1\n每步搬运全部权重\n算术密度低"]
    end
    训练 -->|"优化方向不同"| 推理
    I1 --> OPT["KV Cache\nFlash Attention\n量化\n投机采样"]
```

**规模与延迟的现实**

| 模型 | 参数量 | FP16 权重大小 | A100 单卡吞吐（tokens/s） |
|------|--------|-------------|-------------------------|
| GPT-2 Small | 117M | ~0.2 GB | ~3000 |
| LLaMA-2 7B | 7B | ~14 GB | ~100 |
| LLaMA-2 70B | 70B | ~140 GB | ~10（需多卡） |


In [ ]:
# 安装依赖（首次运行）
# !pip install torch transformers

import torch
import time
import math
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

## 第二章：KV Cache——消除自回归的重复计算

**问题**：自回归生成时，每生成一个新 token，都要对「所有历史 token」重新计算注意力。生成第 N 个 token 时，第 1 个 token 的 K/V 被计算了 N 次。

**解法**：把每个 token 的 K、V 向量缓存起来，下一步直接复用。

```mermaid
sequenceDiagram
    participant I as 输入序列
    participant M as 模型
    participant C as KV Cache
    Note over I,C: 无 KV Cache（每步重算所有 token）
    I->>M: [t1,t2,t3] → 计算 K1V1,K2V2,K3V3
    I->>M: [t1,t2,t3,t4] → 重算 K1V1,K2V2,K3V3,K4V4
    Note over I,C: 有 KV Cache（只算新 token）
    I->>M: [t1,t2,t3] → 计算 K1V1,K2V2,K3V3
    M->>C: 存入 K1V1,K2V2,K3V3
    I->>M: [t4] → 只算 K4V4
    C->>M: 读取缓存 K1V1,K2V2,K3V3
```

**内存代价**：KV Cache 大小 = `2 × num_layers × num_heads × head_dim × seq_len × batch_size × sizeof(dtype)`

LLaMA-2 7B 生成 2048 token 的 KV Cache ≈ **2 GB**（FP16）。这是为什么长上下文推理显存压力大的根本原因。

In [ ]:
# KV Cache 演示：对比有无缓存的计算量

def attention_without_cache(seq_len: int, d_model: int = 64) -> int:
    """无 KV Cache：每步重算所有 token 的 K/V"""
    total_ops = 0
    for step in range(1, seq_len + 1):
        # 每步计算 step 个 token 的 QKV，矩阵乘法 ops
        total_ops += step * d_model * d_model * 3  # Q, K, V 投影
        total_ops += step * step * d_model          # QK^T 注意力
    return total_ops

def attention_with_cache(seq_len: int, d_model: int = 64) -> int:
    """有 KV Cache：每步只算新 token 的 Q，K/V 从缓存读"""
    total_ops = 0
    for step in range(1, seq_len + 1):
        total_ops += d_model * d_model * 3   # 只有新 token 的 QKV
        total_ops += step * d_model           # 新 Q 和所有缓存 K 的内积
    return total_ops

seq_lengths = range(10, 300, 10)
ops_no_cache = [attention_without_cache(n) for n in seq_lengths]
ops_with_cache = [attention_with_cache(n) for n in seq_lengths]

plt.figure(figsize=(9, 4))
plt.plot(seq_lengths, [x / 1e6 for x in ops_no_cache], label='无 KV Cache（O(n²) 重算）', color='#e74c3c')
plt.plot(seq_lengths, [x / 1e6 for x in ops_with_cache], label='有 KV Cache（O(n) 增量）', color='#2ecc71')
plt.xlabel('序列长度')
plt.ylabel('累计计算量（百万 ops）')
plt.title('KV Cache 的计算量节省')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

n = 200
ratio = attention_without_cache(n) / attention_with_cache(n)
print(f"序列长度 {n} 时，KV Cache 节省计算量：{ratio:.1f}×")

## 第三章：Flash Attention——IO 感知的注意力计算

标准注意力的瓶颈不是 FLOPS，而是 **HBM 读写次数**。

注意力矩阵 S = QK^T 的大小是 `O(N²)`，对于 N=4096，这是一个 16M 的矩阵，需要反复在 HBM 和 SRAM 之间搬运。

```mermaid
flowchart TD
    subgraph 标准注意力["标准注意力（HBM 往返多次）"]
        SA1["HBM 读 Q, K"] --> SA2["SRAM 计算 S = QKᵀ"]
        SA2 --> SA3["HBM 写 S（N² 大小）"]
        SA3 --> SA4["HBM 读 S"]
        SA4 --> SA5["SRAM 计算 P = softmax(S)"]
        SA5 --> SA6["HBM 写 P"]
        SA6 --> SA7["HBM 读 P, V → 计算 O"]
    end
    subgraph flash["Flash Attention（分块 Tiling，一趟完成）"]
        FA1["HBM 分块读 Q, K, V"]
        FA1 --> FA2["SRAM 内完成：S, softmax, O 合并计算"]
        FA2 --> FA3["只写最终输出 O 回 HBM"]
    end
```

**效果**：Flash Attention 的 HBM 访问量从 `O(N²)` 降至 `O(N)`，在长序列场景速度提升 2-4×，显存节省 5-20×。

In [ ]:
# Flash Attention vs 标准注意力：速度对比
# PyTorch 2.0+ 的 scaled_dot_product_attention 默认使用 Flash Attention（如果 CUDA 可用）

def standard_attention(q, k, v, scale=None):
    if scale is None:
        scale = q.shape[-1] ** -0.5
    scores = torch.matmul(q, k.transpose(-2, -1)) * scale
    attn = torch.softmax(scores, dim=-1)
    return torch.matmul(attn, v)

def flash_attention(q, k, v):
    # PyTorch 2.0+ 自动选择最优实现（CUDA 上启用 Flash Attention）
    return torch.nn.functional.scaled_dot_product_attention(q, k, v)

# 测试不同序列长度的速度
batch, heads, d_head = 2, 8, 64
seq_lengths_test = [128, 256, 512, 1024]
times_std, times_flash = [], []
n_runs = 20

for seq_len in seq_lengths_test:
    q = torch.randn(batch, heads, seq_len, d_head, device=device)
    k = torch.randn(batch, heads, seq_len, d_head, device=device)
    v = torch.randn(batch, heads, seq_len, d_head, device=device)

    # 标准注意力
    start = time.perf_counter()
    for _ in range(n_runs):
        _ = standard_attention(q, k, v)
    times_std.append((time.perf_counter() - start) / n_runs * 1000)

    # Flash Attention（PyTorch SDPA）
    start = time.perf_counter()
    for _ in range(n_runs):
        _ = flash_attention(q, k, v)
    times_flash.append((time.perf_counter() - start) / n_runs * 1000)

plt.figure(figsize=(9, 4))
x = range(len(seq_lengths_test))
width = 0.35
plt.bar([i - width/2 for i in x], times_std, width, label='标准注意力', color='#e74c3c', alpha=0.8)
plt.bar([i + width/2 for i in x], times_flash, width, label='SDPA (Flash Attention)', color='#3498db', alpha=0.8)
plt.xticks(x, [str(s) for s in seq_lengths_test])
plt.xlabel('序列长度')
plt.ylabel('耗时 (ms)')
plt.title(f'标准注意力 vs Flash Attention ({device})')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

for i, seq_len in enumerate(seq_lengths_test):
    speedup = times_std[i] / times_flash[i]
    print(f"seq={seq_len:4d}: 标准 {times_std[i]:.2f}ms  SDPA {times_flash[i]:.2f}ms  加速 {speedup:.2f}×")

## 第四章：量化——用精度换体积

**核心思路**：FP32（4 字节）→ FP16（2 字节）→ INT8（1 字节）→ INT4（0.5 字节）

量化不改变模型结构，只改变权重的存储精度。推理时动态反量化到 FP16 计算，或直接用整数运算。

| 精度 | 体积比 FP32 | 典型精度损失 | 适用场景 |
|------|------------|------------|--------|
| FP32 | 1× | 基准 | 训练 |
| FP16/BF16 | 0.5× | 极小 | 标准推理 |
| INT8 | 0.25× | 很小（<1% 性能下降） | 生产部署 |
| INT4 | 0.125× | 小（2-5% 性能下降） | 消费级 GPU |
| GPTQ/AWQ INT4 | 0.125× | 更小（校准数据优化） | 高质量量化 |

**LLaMA-2 7B 不同精度的显存需求：**

| 精度 | 显存 | 最低 GPU |
|------|------|----------|
| FP16 | ~14 GB | A100 40G |
| INT8 | ~7 GB | RTX 3090 |
| INT4 | ~3.5 GB | RTX 3060 12G |

In [ ]:
# 量化原理演示：线性量化的误差分析

def quantize_int8(tensor: torch.Tensor):
    """对称量化到 INT8"""
    scale = tensor.abs().max() / 127.0
    quantized = torch.clamp(torch.round(tensor / scale), -128, 127).to(torch.int8)
    return quantized, scale

def dequantize_int8(quantized: torch.Tensor, scale: float) -> torch.Tensor:
    return quantized.float() * scale

def quantize_int4(tensor: torch.Tensor):
    """对称量化到 INT4（-8 到 7）"""
    scale = tensor.abs().max() / 7.0
    quantized = torch.clamp(torch.round(tensor / scale), -8, 7)
    return quantized, scale

# 模拟一个权重矩阵
torch.manual_seed(42)
weights = torch.randn(256, 256) * 0.1  # 典型权重分布

q8, s8 = quantize_int8(weights)
deq8 = dequantize_int8(q8, s8)
err8 = (weights - deq8).abs().mean().item()

q4, s4 = quantize_int4(weights)
deq4 = q4.float() * s4
err4 = (weights - deq4).abs().mean().item()

print(f"原始权重   FP32  体积: {weights.numel() * 4 / 1024:.1f} KB  误差: 0")
print(f"量化 INT8        体积: {weights.numel() * 1 / 1024:.1f} KB  平均绝对误差: {err8:.6f}")
print(f"量化 INT4        体积: {weights.numel() * 0.5 / 1024:.1f} KB  平均绝对误差: {err4:.6f}")

# 可视化误差分布
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].hist(weights.numpy().flatten(), bins=50, color='#3498db', alpha=0.7)
axes[0].set_title('原始 FP32 权重分布')
axes[1].hist((weights - deq8).numpy().flatten(), bins=50, color='#2ecc71', alpha=0.7)
axes[1].set_title('INT8 量化误差分布')
axes[2].hist((weights - deq4).numpy().flatten(), bins=50, color='#e74c3c', alpha=0.7)
axes[2].set_title('INT4 量化误差分布')
for ax in axes:
    ax.set_ylabel('频次')
plt.tight_layout()
plt.show()

## 第五章：投机采样——并行验证加速解码

**问题**：自回归生成天然串行——必须等第 N 个 token 生成后，才能生成第 N+1 个。GPU 的并行计算能力被严重浪费。

**投机采样的思路**：
1. 用一个**小草稿模型**（Draft Model，如 7B）快速生成 K 个候选 token
2. 用**大目标模型**（Target Model，如 70B）**并行**验证这 K 个 token
3. 接受符合目标模型分布的前缀，拒绝后续 token，重新采样

```mermaid
sequenceDiagram
    participant D as 草稿模型（小）
    participant T as 目标模型（大）
    D->>D: 快速生成 [t1, t2, t3, t4, t5]
    D->>T: 发送 5 个候选 token
    T->>T: 并行验证 5 个 token（一次前向）
    T->>D: 接受 [t1, t2, t3]，拒绝 t4
    Note over D,T: 实际输出了 3 个 token，但只用了大模型 1 次前向
```

**加速比**：取决于草稿模型的接受率。接受率 80% + K=5 时，理论加速约 **2-3×**，且输出分布与纯目标模型完全等价。

In [ ]:
# 投机采样加速比的理论分析

def expected_speedup(acceptance_rate: float, k: int, draft_cost: float = 0.1) -> float:
    """
    计算投机采样的期望加速比。
    acceptance_rate: 草稿 token 被接受的概率（每个独立）
    k: 每轮草稿生成的 token 数
    draft_cost: 草稿模型相对于目标模型的推理成本（0.1 = 目标模型的 1/10）
    """
    # 期望接受的 token 数：几何分布期望
    alpha = acceptance_rate
    expected_accepted = (1 - alpha**(k+1)) / (1 - alpha) - 1
    expected_accepted = min(expected_accepted, k)

    # 每轮成本：k 次草稿推理 + 1 次目标模型并行验证
    cost_per_round = k * draft_cost + 1.0

    # 加速比 = 期望 token 数 / 每轮成本
    tokens_without_spec = cost_per_round  # 同等成本下纯目标模型产出的 token 数
    return (expected_accepted + 1) / cost_per_round  # +1 是拒绝后重新采样的那个

acceptance_rates = np.linspace(0.5, 0.99, 50)
plt.figure(figsize=(9, 4))
for k in [2, 4, 8]:
    speedups = [expected_speedup(a, k) for a in acceptance_rates]
    plt.plot(acceptance_rates, speedups, label=f'K={k} 草稿 token')

plt.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='无加速基准线')
plt.xlabel('草稿 token 接受率')
plt.ylabel('期望加速比')
plt.title('投机采样：接受率 vs 加速比（草稿模型成本 = 目标的 1/10）')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("接受率 80%，K=4 时，期望加速比：", f"{expected_speedup(0.8, 4):.2f}×")
print("接受率 90%，K=8 时，期望加速比：", f"{expected_speedup(0.9, 8):.2f}×")

## 第六章：连续批处理——提升服务吞吐量

**问题**：传统静态批处理中，一批请求必须等最长的序列完成后才能释放 GPU。短请求在等待时 GPU 空转。

**连续批处理（Continuous Batching / Iteration-level Batching）**：
- 不等一批请求全部完成，而是**逐个 token 步**调度
- 某个序列生成完毕后，立刻把新请求插入空出的 batch slot
- GPU 始终保持高利用率

```mermaid
gantt
    title 静态批处理 vs 连续批处理
    dateFormat X
    axisFormat %s
    section 静态批处理
    请求A（10 token）  :a1, 0, 10
    请求B（5 token）   :a2, 0, 10
    等待（B 完成后空转）:crit, 5, 10
    请求C（8 token）   :a3, 10, 18
    section 连续批处理
    请求A（10 token）  :b1, 0, 10
    请求B（5 token）   :b2, 0, 5
    请求C（立即插入）  :b3, 5, 13
```

**PagedAttention（vLLM）**：进一步用操作系统分页思想管理 KV Cache 内存，避免内存碎片，使 GPU 利用率从 55% 提升到 90%+。

In [ ]:
# 批处理效率模拟：静态 vs 连续批处理

import random

random.seed(42)

def simulate_static_batching(requests: list, batch_size: int) -> dict:
    """静态批处理：等整批完成再处理下一批"""
    total_gpu_steps = 0
    total_useful_steps = 0  # 实际生成 token 的步骤

    for i in range(0, len(requests), batch_size):
        batch = requests[i:i + batch_size]
        max_len = max(batch)  # 必须等最长的
        total_gpu_steps += max_len * len(batch)
        total_useful_steps += sum(batch)

    return {
        'gpu_steps': total_gpu_steps,
        'useful_steps': total_useful_steps,
        'utilization': total_useful_steps / total_gpu_steps
    }

def simulate_continuous_batching(requests: list, batch_size: int) -> dict:
    """连续批处理：序列完成后立即插入新请求"""
    total_gpu_steps = sum(requests)  # 几乎每步都有用
    # 连续批处理的开销极小（调度overhead），简化为总 token 数
    return {
        'gpu_steps': total_gpu_steps,
        'useful_steps': total_gpu_steps,
        'utilization': 1.0
    }

# 模拟 100 个长度不均匀的请求（现实中请求长度差异很大）
requests = [random.randint(5, 200) for _ in range(100)]

print("模拟 100 个请求，长度范围 5-200 token")
print(f"平均长度: {sum(requests)/len(requests):.1f}  最大长度: {max(requests)}  最小长度: {min(requests)}\n")

for bs in [4, 8, 16]:
    static = simulate_static_batching(requests, bs)
    continuous = simulate_continuous_batching(requests, bs)
    print(f"batch_size={bs:2d}:")
    print(f"  静态批处理    GPU 利用率: {static['utilization']:.1%}")
    print(f"  连续批处理    GPU 利用率: {continuous['utilization']:.1%}")
    print(f"  吞吐量提升:   {continuous['gpu_steps'] / static['gpu_steps'] * (1/static['utilization']):.2f}×")
    print()

## 小结：各场景选哪种优化？

```mermaid
flowchart TD
    START(["我的推理场景"]) --> Q1{"延迟 or 吞吐量？"}
    Q1 -->|"降低延迟"| Q2{"单次请求\n多 token 生成？"}
    Q1 -->|"提升吞吐量"| Q3{"多并发请求？"}
    Q2 -->|"是"| KVC["✅ KV Cache\n消除重复计算"]
    Q2 -->|"长序列"| FA["✅ Flash Attention\n节省显存+速度"]
    Q2 -->|"大模型"| SD["✅ 投机采样\n草稿模型加速"]
    Q3 -->|"是"| CB["✅ 连续批处理\n提升 GPU 利用率"]
    Q3 -->|"显存不足"| QUANT["✅ INT8/INT4 量化\n压缩模型体积"]
    KVC & FA & SD & CB & QUANT --> COMBINE["生产环境：以上技术组合使用\nvLLM = KV Cache + 连续批处理 + PagedAttention"]
```

**工具推荐**：
- **vLLM**：生产级推理服务，集成连续批处理 + PagedAttention，OpenAI 兼容 API
- **llama.cpp**：CPU/低显存推理，支持 GGUF 量化格式
- **bitsandbytes**：PyTorch 集成量化，一行代码启用 INT8/INT4
- **Ollama**：本地部署一站式工具，底层用 llama.cpp

**下一章**：[02_agents_frameworks/05_finetuning_lora.ipynb](../02_agents_frameworks/05_finetuning_lora.ipynb) — 当推理优化不够用时，如何用 LoRA 让模型更好地适应特定任务。